**Training**

In [1]:
!rm -rf /kaggle/working/SwinIR

In [2]:
#!git clone https://github.com/kavindamihiran/SwinIR
#!git clone -b kavinda1 https://github.com/kavindamihiran/SwinIR.git
!git clone -b swinllie https://github.com/kavindamihiran/SwinIR.git

Cloning into 'SwinIR'...
remote: Enumerating objects: 1748, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 1748 (delta 10), reused 15 (delta 7), pack-reused 1726 (from 2)
Receiving objects: 100% (1748/1748), 1.08 GiB | 44.91 MiB/s, done.
Resolving deltas: 100% (196/196), done.
Updating files: 100% (1385/1385), done.
Filtering content: 100% (2/2), 78.31 MiB | 18.35 MiB/s, done.


In [3]:
%cd /kaggle/working/SwinIR

/kaggle/working/SwinIR


In [4]:
!pip install -r requirements.txt

In [5]:
# import yaml
# import os

# config_file_path = '/content/SwinIR/configs/swinllie_lol.yaml'

# # Check if the file exists before attempting to read
# if not os.path.exists(config_file_path):
#     print(f"Error: Configuration file not found at {config_file_path}")
# else:
#     with open(config_file_path, 'r') as file:
#         config = yaml.safe_load(file)

#     # Modify the config settings
#     config['training']['epochs'] = 150
#     config['resume']['enabled'] = True
#     config['resume']['checkpoint_path'] = '/content/SwinIR/experiments/test_run/checkpoints/final.pth'

#     # Write the modified config back to the file
#     with open(config_file_path, 'w') as file:
#         yaml.safe_dump(config, file, default_flow_style=False)

#     print(f"Successfully updated {config_file_path}")

In [6]:
!nvidia-smi

Sat Apr 25 17:29:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!python train.py --config configs/swinllie_lol.yaml

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [8]:
 !rm -rf /kaggle/working/best.pth

In [9]:
!cp /kaggle/working/SwinIR/experiments/test_run/checkpoints/best.pth /kaggle/working/


## Step 1: Efficiency Metrics (GFLOPs and Inference Time)
Calculates GFLOPs and FPS for a standard resolution (400x600) to prove the model's efficiency compared to heavier architectures.

In [10]:
!pip install thop
import torch
import time
from thop import profile
import sys
sys.path.append('/kaggle/working/SwinIR')
from swinllie import SwinLLIE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SwinLLIE(img_size=128, embed_dim=60, depths=[4, 4, 4], num_heads=[6, 6, 6], window_size=8).to(device)
model.eval()

# Dummy input at 400x600 resolution
dummy_input = torch.randn(1, 3, 400, 600).to(device)

# GFLOPs calculation
flops, params = profile(model, inputs=(dummy_input, ), verbose=False)
print(f"Params: {params / 1e6:.2f} M")
print(f"GFLOPs: {flops / 1e9:.2f} G")

# Inference Time calculation (FPS)
with torch.no_grad():
    # Warmup
    for _ in range(10):
        _ = model(dummy_input)
        
    torch.cuda.synchronize()
    start_time = time.time()
    num_iterations = 50
    for _ in range(num_iterations):
        _ = model(dummy_input)
    torch.cuda.synchronize()
    end_time = time.time()

avg_time = (end_time - start_time) / num_iterations
print(f"Inference Time: {avg_time * 1000:.2f} ms")
print(f"FPS: {1.0 / avg_time:.2f}")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Params: 4.68 M
GFLOPs: 267.67 G
Inference Time: 1361.53 ms
FPS: 0.73


## Step 2: Ablation Study
Trains the model with different loss configurations to prove the effectiveness of each component, especially the Exposure Control Loss.

In [11]:
import yaml
import os

config_file_path = '/kaggle/working/SwinIR/configs/swinllie_lol.yaml'

def run_ablation(name, lambda_l1=0.0, lambda_vgg=0.0, lambda_color=0.0, lambda_exposure=0.0):
    print(f"\n--- Running Ablation: {name} ---")
    with open(config_file_path, 'r') as file:
        config = yaml.safe_load(file)
        
    config['loss']['lambda_l1'] = lambda_l1
    config['loss']['lambda_vgg'] = lambda_vgg
    config['loss']['lambda_color'] = lambda_color
    config['loss']['lambda_exposure'] = lambda_exposure
    
    config['training']['epochs'] = 50 # Reduced for ablation speed
    config['training']['save_dir'] = f"./experiments/ablation_{name}"
    
    with open(config_file_path, 'w') as file:
        yaml.safe_dump(config, file, default_flow_style=False)
        
    # Run training
    os.system(f"python train.py --config {config_file_path}")

# Run the 4 ablation models
# 1. Base (L1 loss only)
run_ablation('base_l1', lambda_l1=1.0)
# 2. Base + VGG + Color
run_ablation('base_vgg_color', lambda_l1=1.0, lambda_vgg=0.1, lambda_color=0.5)
# 3. Base + All Losses (No Exposure Loss)
run_ablation('no_exposure', lambda_l1=1.0, lambda_vgg=0.1, lambda_color=0.5, lambda_exposure=0.0)
# 4. Ours (Full Loss)
run_ablation('full', lambda_l1=1.0, lambda_vgg=0.1, lambda_color=0.5, lambda_exposure=1.0)


--- Running Ablation: base_l1 ---


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Device: cuda
Found 1 GPU
Config: /kaggle/working/SwinIR/configs/swinllie_lol.yaml
Single GPU setup: batch_size=8, workers=4
Model params: 4,702,503
[LOL-TRAIN] Loaded 485 image pairs
Training samples: 485
Total batch size: 8


Epoch 2/50:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 1: Loss = 0.3506, LR = 0.000200


...

Epoch 48: Loss = 1.2631, LR = 0.000002


Epoch 50/50:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 49: Loss = 1.2621, LR = 0.000001


Epoch 50/50: 100%|██████████| 60/60 [00:19<00:00,  3.03it/s, loss=1.4072]


Epoch 50: Loss = 1.2439, LR = 0.000001
[LOL-TEST] Loaded 15 image pairs
  -> Test PSNR: 17.57 dB, SSIM: 0.8218, Best interval loss: 1.2037

Training complete! Best PSNR: 17.76 dB, SSIM: 0.8234, Loss: 1.2368


## Step 3: Evaluate on LOL-v2 Dataset
Evaluates the trained model on the LOL-v2 dataset to prove generalization across different datasets.
**Note**: Make sure to add the `tanhyml/lol-v2-dataset` dataset to this Kaggle notebook.

In [10]:
import os

# Search for LOL-v2 dataset path in Kaggle
lol_v2_path = '/kaggle/input/lol-v2-dataset/LOL-v2'
if not os.path.exists(lol_v2_path):
    lol_v2_path = '/kaggle/input/datasets/tanhyml/lol-v2-dataset/LOL-v2'
    
if os.path.exists(lol_v2_path):
    print("Evaluating on LOL-v2 dataset...")
    
    # We use 'cd /kaggle/working/SwinIR' in every command to ensure Jupyter's CWD state doesn't break relative paths
    !cd /kaggle/working/SwinIR && mkdir -p ./datasets/LOL-v2/test/low ./datasets/LOL-v2/test/high
    !cd /kaggle/working/SwinIR && cp -r {lol_v2_path}/Real_captured/Test/Low/* ./datasets/LOL-v2/test/low/
    
    # The ground truth folder is called 'Normal' in LOL-v2
    !cd /kaggle/working/SwinIR && cp -r {lol_v2_path}/Real_captured/Test/Normal/* ./datasets/LOL-v2/test/high/
    
    # Run inference and check metrics
    !cd /kaggle/working/SwinIR && python inference.py --input ./datasets/LOL-v2/test/low --output ./results_lolv2 --weights ./experiments/test_run/checkpoints/best.pth
    
    # Note: For check_metrics.py, we temporarily change dataset_type to 'generic' to avoid LOL's eval15 hardcoding
    !cd /kaggle/working/SwinIR && sed -i "s/get_dataloader('lol'/get_dataloader('generic'/g" check_metrics.py
    !cd /kaggle/working/SwinIR && python check_metrics.py --dataset ./datasets/LOL-v2 --weights ./experiments/test_run/checkpoints/best.pth
else:
    print("Please click 'Add Data' on the right panel and search for 'tanhyml/lol-v2-dataset' to attach it.")

Evaluating on LOL-v2 dataset...
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotat

In [17]:
import os
import shutil
import re

# 1. Automatically hunt for the LOL-v2 dataset path
search_paths = [
    '/kaggle/input/lol-v2-dataset/LOL-v2/Real_captured/Test',
    '/kaggle/input/datasets/tanhyml/lol-v2-dataset/LOL-v2/Real_captured/Test',
    '/kaggle/input/lol-v2/Real_captured/Test',
    '/kaggle/input/datasets/tanhyml/lol-v2-dataset/Real_captured/Test'
]

valid_path = None
for p in search_paths:
    if os.path.exists(p):
        valid_path = p
        break

if valid_path:
    print(f"Found LOL-v2 test data at: {valid_path}")
    
    # 2. Setup target directories
    low_target = '/kaggle/working/SwinIR/datasets/LOL-v2/test/low'
    high_target = '/kaggle/working/SwinIR/datasets/LOL-v2/test/high'
    os.makedirs(low_target, exist_ok=True)
    os.makedirs(high_target, exist_ok=True)
    
    # 3. Find source directories
    dirs = os.listdir(valid_path)
    low_src = next((os.path.join(valid_path, d) for d in dirs if d.lower() == 'low'), None)
    high_src = next((os.path.join(valid_path, d) for d in dirs if d.lower() in ['normal', 'high']), None)
    
    if low_src and high_src:
        low_files = os.listdir(low_src)
        high_files = os.listdir(high_src)
        
        # 4. Copy and RENAME files so they match exactly (e.g., 'low001.png' -> '001.png')
        for f in low_files:
            num = re.search(r'\d+', f)
            if num:
                new_name = f"{num.group()}.png"
                shutil.copy(os.path.join(low_src, f), os.path.join(low_target, new_name))
                
        for f in high_files:
            num = re.search(r'\d+', f)
            if num:
                new_name = f"{num.group()}.png"
                shutil.copy(os.path.join(high_src, f), os.path.join(high_target, new_name))
            
        print(f"Successfully copied and renamed {len(low_files)} pairs of images!")
        
        # 5. Run the evaluation!
        print("\n--- Running Evaluation ---")
        !cd /kaggle/working/SwinIR && sed -i "s/map_location='cpu')/map_location='cpu', weights_only=False)/g" check_metrics.py
        !cd /kaggle/working/SwinIR && sed -i "s/get_dataloader('lol'/get_dataloader('generic'/g" check_metrics.py
        
        !cd /kaggle/working/SwinIR && python check_metrics.py --dataset ./datasets/LOL-v2 --weights /kaggle/working/best.pth
    else:
        print("Error: Could not find the 'Low' or 'Normal' folders inside the dataset!")
else:
    print("Error: Could not find the dataset at all! Make sure you clicked 'Add Data' and attached it.")


Found LOL-v2 test data at: /kaggle/input/datasets/tanhyml/lol-v2-dataset/LOL-v2/Real_captured/Test
Successfully copied and renamed 100 pairs of images!

--- Running Evaluation ---
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-

## Step 5: Implementation Details (For the paper)
- **GPU Hardware**: Display the exact GPU used in Kaggle (usually 1x NVIDIA Tesla T4 or P100).
- **Software Stack**: PyTorch 2.0+ with CUDA.
Be sure to record the training time displayed above to add to your paper!

In [13]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Sat Apr 25 19:13:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             34W /   70W |    1955MiB /  15360MiB |     54%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----